Take a trained wandb run, pick the checkpoint that minimizes rollout **VRMSE**
(`rollout_valid_*/full_VRMSE_T=all_mean`), and evaluate it on a **different** Well dataset.

Outputs:
- **`zero_shot(project, run, data=...)`** — one run; optional original-test comparison
- **`compare_zero_shot(project, [run, ...], data=...)`** — table of zero-shot
  **rollout_test VRMSE** for the best-by-rollout checkpoint of each run (no original-test re-eval)


Tables use **VRMSE** (cross-run comparable). Avoid the raw wandb scalars `valid` /
`rollout_valid` / `test` / `rollout_test` when comparing CRPS to det — those follow
each run's train `loss_fn` (MAE vs CRPS).

Requires the target data to already be a Well tree (`data/{train,valid,test}` +
`stats.yaml` / `metadata`).

### Inference cache

Every evaluated rollout is saved once under `demo_notebooks/_analysis_cache`, with a
separate folder per wandb **entity / project / model / run / dataset / split**.
Entries also fingerprint the checkpoint path, data files, and inference settings.
The VRMSE table, videos, prediction distributions, and RMS plots all reuse those
physical-unit trajectories. After `compare_zero_shot` has populated a dataset,
editing or rerunning its plot cells does **not** load the model or run inference.
A completed cache is never overwritten unless you set `REFRESH_CACHE=True`.
Older caches (schema v1, flatter paths) are ignored; the first eval after this
change rebuilds each project/dataset/model once.


In [ ]:
%load_ext autoreload
%autoreload 2

import pathlib
import sys

_cwd = pathlib.Path.cwd().resolve()
REPO_ROOT = next(
    p for p in (_cwd, *_cwd.parents) if (p / "walrus" / "__init__.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))

from walrus.analysis import (
    PANEL_SIZE,
    allows_missing_checkpoint,
    annotation_font_size,
    apply_paper_style,
    cached_rollouts,
    collect_prediction_values,
    compare_flow_metrics,
    compare_paper_residual,
    compare_zero_shot,
    evaluate_flow_metrics,
    get_run,
    list_data_configs,
    load_data_config,
    panel_figure,
    panel_grid_size,
    plot_rms_velocity_by_group,
    plot_rms_velocity_multi_run,
    plot_rms_velocity_overlay,
    run_history,
    select_checkpoint,
    spectral_metrics_from_rollouts,
    vrmse_from_rollouts,
    zero_shot,
)

## Configure

Edit these cells, then run the peek + evaluate sections.

In [ ]:
# --- wandb run to take the checkpoint from ---
PROJECT = "morphogenesis_no_myosin"
ENTITY = None  # None -> your default wandb entity

# Display name or run id
RUN = "Walrus_crps_morph_myosin_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001"

# Multi-run zero-shot table (compare_zero_shot). Use wandb ids if names collide.
# Advection / Mean-field have no learned weights (or only an EMA buffer). They
# are included here so the VRMSE / spectral bars match analyze_checkpoints.
RUNS = [
    "FFNO_morph_WT-morph-delta-FFNOW[--]-AdamW-0.0001",
    "PoseidonL_ft_morph_WT-morph-full-ScOTW[--]-AdamW-0.0001",
    "SineNet_morph_WT-morph-delta-SineN[--]-AdamW-0.0001",
    "Walrus_crps_morph_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001",
    "Walrus_ft_morph_WT_no_myosin-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001",
    "Advection_morph_WT-morph-full-Advec[--]-AdamW-0.0001",
    "MeanField_morph_WT-morph-full-MeanF[--]-AdamW-0.0001",
]

# Set only if checkpoints were moved from the path in the run config
CHECKPOINTS_DIR = None

# --- dataset to zero-shot on (Hydra config under walrus/configs/data/) ---
DATA = "morphogenesis_WT_17_degrees"
# Dataset to treat as "original" for the comparison table. Required when the
# run's training path (e.g. processed_morphodynamic_atlas) is no longer on disk.
ORIGINAL_DATA = "morphogenesis_WT"

# Evaluation knobs
FULL = True          # False = short validation subset (faster smoke test)
MAKE_VIDEOS = True
BATCH_SIZE = 1
# None = keep the n_steps_input the run was trained with (recommended)
N_STEPS_INPUT = None
# Also re-eval the same ckpt on the run's original test set for a comparison table
COMPARE_TO_ORIGINAL = True
# How many rollout videos to write (trainer default is 3)
NUM_DETAILED_LOGS = 5

# Look of every rollout video written from the cache: the diverging, auto-centered
# scheme used for the myosin figure videos, panels flush and colorbar-free, with the
# field / row names in a margin around the grid instead of over the data.
# "labels": "inside" draws them on the frames, "none" omits them; "layout": "panels"
# brings back per-field colorbars; rows=("true", "pred") drops the |error| row; cmap
# takes any diverging map (see DIVERGING_COLORMAPS). Restyling only re-encodes video
# — it never re-runs inference.
VIDEO_STYLE = {
    "layout": "grid",
    "cmap": "RdBu_r",
    "error_cmap": "magma",
    "percentile": 97.0,
    "center": "auto",
    "labels": "outside",
    "rows": ("true", "pred", "error"),
}

# Every rollout is saved here once. Tables, videos, distributions and RMS plots all
# read the same cache; editing/re-running plots does not execute the model again.
CACHE_DIR = REPO_ROOT / "demo_notebooks" / "_analysis_cache"
REFRESH_CACHE = False  # True = deliberately rerun inference and replace this cache key
print(f"rollout cache: {CACHE_DIR}")

print("available data configs:")
print([c for c in list_data_configs() if "morph" in c.lower() or c in (DATA,)])
print("...", len(list_data_configs()), "total")

In [ ]:
import wandb
from walrus.analysis.checkpoint_analysis import (
    allows_missing_checkpoint,
    available_checkpoints,
)
import pathlib

api = wandb.Api()
entity = ENTITY or api.default_entity
wb_runs = api.runs(f"{entity}/{PROJECT}")

# optional: only finished runs that actually trained
#wb_runs = [r for r in wb_runs if r.state == "finished"]


RUNS = sorted(
    {
        r.name
        for r in wb_runs
        if available_checkpoints(pathlib.Path(r.config["checkpoint"]["save_dir"]))
        or allows_missing_checkpoint(r)
    }
)
print(len(RUNS), "runs")
print("\n".join(RUNS))

## Peek (cheap)

Confirm which checkpoint will be loaded and which data path will be used, before the
expensive eval.

In [ ]:
run = get_run(PROJECT, RUN, ENTITY)
sel = select_checkpoint(run, metric="rollout_valid", checkpoints_dir=CHECKPOINTS_DIR)
print(f"run:        {run.name} ({run.id})")
print(f"checkpoint: epoch {sel.epoch}  rollout_valid_VRMSE={sel.value:.4f}")
print(f"            {sel.path}")

data_cfg = load_data_config(DATA)
info = data_cfg.module_parameters.well_dataset_info
print(f"\ndata:       {DATA}")
for name, meta in info.items():
    print(f"  {name}: {meta.get('path')}")

hist = run_history(run)
hist.tail(8)


In [ ]:
from walrus.analysis import evaluate_checkpoint, get_run

res = evaluate_checkpoint(
    get_run(PROJECT, RUN, ENTITY),
    data=DATA,
    splits=("rollout_test",),
    full=FULL,
    data_overrides={"batch_size": BATCH_SIZE},
    make_videos=MAKE_VIDEOS,
    num_detailed_logs=NUM_DETAILED_LOGS,
    video_style=VIDEO_STYLE,
    cache_dir=CACHE_DIR,
    checkpoints_dir=CHECKPOINTS_DIR,
    refresh_cache=REFRESH_CACHE,
)
print("cache_hit:", res["cache_hit"])
print("rollout_test VRMSE:", res["rollout_test"])
for path in res["videos"]:
    print(path)

## Run zero-shot eval (+ original comparison)

Loads the best-by-rollout-VRMSE checkpoint, evaluates **test** / **rollout_test VRMSE** on
the run's original test set and on `DATA`, and shows a comparison table. Videos are
only written for the zero-shot leg, rendered from the cache in `VIDEO_STYLE` and stored
under `<cache entry>/videos/<layout>_<cmap>/`. Changing `VIDEO_STYLE` re-encodes them
from the cached arrays; it never re-runs the model.


In [ ]:
results = zero_shot(
    project=PROJECT,
    run=RUN,
    data=DATA,
    entity=ENTITY,
    full=FULL,
    make_videos=MAKE_VIDEOS,
    checkpoints_dir=CHECKPOINTS_DIR,
    batch_size=BATCH_SIZE,
    n_steps_input=N_STEPS_INPUT,
    compare_to_original=COMPARE_TO_ORIGINAL,
    original_data=ORIGINAL_DATA,
    num_detailed_logs=NUM_DETAILED_LOGS,
    cache_dir=CACHE_DIR,
    refresh_cache=REFRESH_CACHE,
    video_style=VIDEO_STYLE,
)
results["comparison"]

## Prediction distributions: original vs zero-shot

Overlay histograms of **model predictions** and **ground truth** on the original test
set vs the zero-shot dataset (same best-by-rollout-VRMSE checkpoint).

With `SPLIT="rollout_test"`, dens. are plotted at rollout horizons **5 / 10 / 20 / 30 / full**
frames (first N predicted frames, pooled over pixels × those frames — not a time-average).
Pred and GT panels for the same field share x/y axis limits within each horizon.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from pathlib import Path

%matplotlib inline

apply_paper_style()

# One-step: SPLIT="test". Rollout: SPLIT="rollout_test" (slower).
# For rollout, dens. are pooled over the first N frames at each horizon (not time-averaged).
SPLIT = "rollout_test"
FIELDS = ["velocity_x", "velocity_y"]  # Walrus / isotropic
# or FIELDS = None  # plot every cached fiel  # None = all fields
MAX_BATCHES = None  # e.g. 20 for a quicker smoke plot
# None = full rollout. Ignored when SPLIT == "test" (single step).
HORIZONS = [15, None]
# Shared x-range per field. Widen toward (0, 1) to show more of the tails; tighten
# (e.g. 0.01/0.99) when heavy tails squeeze the bulk into a single hairline bin.
CLIP_QUANTILES = (0.001, 0.999)
# One colour per dataset, reused across the prediction and ground-truth columns so a
# single two-entry figure legend explains the whole grid.
ORIG_COLOR, ZS_COLOR = "#4C78A8", "#F58518"

run = get_run(PROJECT, RUN, ENTITY)
overrides = {"batch_size": BATCH_SIZE}
if N_STEPS_INPUT is not None:
    overrides["n_steps_input"] = N_STEPS_INPUT

horizon_kw = {"horizons": HORIZONS} if SPLIT == "rollout_test" else {}
orig_by_h = collect_prediction_values(
    run,
    data=ORIGINAL_DATA,
    data_overrides=overrides,
    split=SPLIT,
    checkpoints_dir=CHECKPOINTS_DIR,
    max_batches=MAX_BATCHES,
    fields=FIELDS,
    cache_dir=CACHE_DIR,
    refresh_cache=REFRESH_CACHE,
    **horizon_kw,
)
zs_by_h = collect_prediction_values(
    run,
    data=DATA,
    data_overrides=overrides,
    split=SPLIT,
    checkpoints_dir=CHECKPOINTS_DIR,
    max_batches=MAX_BATCHES,
    fields=FIELDS,
    cache_dir=CACHE_DIR,
    refresh_cache=REFRESH_CACHE,
    **horizon_kw,
)

# Normalize to {horizon_key: dist_dict} whether or not horizons= was used.
if SPLIT == "rollout_test":
    horizon_keys = ["full" if h is None else int(h) for h in HORIZONS]
else:
    orig_by_h = {"1-step": orig_by_h}
    zs_by_h = {"1-step": zs_by_h}
    horizon_keys = ["1-step"]

fields = FIELDS or sorted(
    set(orig_by_h[horizon_keys[0]]["pred"]) | set(zs_by_h[horizon_keys[0]]["pred"])
)
epoch = orig_by_h[horizon_keys[0]]["selection"].epoch


def _dataset_label(name):
    """Drop the shared config prefix so legend entries stay narrow."""
    return str(name).removeprefix("morphogenesis_")

for hkey in horizon_keys:
    orig_dist = orig_by_h[hkey]
    zs_dist = zs_by_h[hkey]
    fig, axes = plt.subplots(
        len(fields), 2, figsize=panel_grid_size(2, len(fields)), squeeze=False
    )
    l1_scores = {}

    for row, field in enumerate(fields):
        all_vals = np.concatenate(
            [
                orig_dist["pred"][field],
                zs_dist["pred"][field],
                orig_dist["ref"][field],
                zs_dist["ref"][field],
            ]
        )
        lo, hi = np.quantile(all_vals, CLIP_QUANTILES)
        bins = np.linspace(lo, hi, 80)

        ax_pred = axes[row, 0]
        ax_gt = axes[row, 1]

        n_pred_wt, _, _ = ax_pred.hist(
            orig_dist["pred"][field], bins=bins, density=True, alpha=0.55, color=ORIG_COLOR
        )
        n_pred_zs, _, _ = ax_pred.hist(
            zs_dist["pred"][field], bins=bins, density=True, alpha=0.55, color=ZS_COLOR
        )
        ax_pred.set_title(f"{field} — predictions")
        ax_pred.set_ylabel("density")

        n_gt_wt, _, _ = ax_gt.hist(
            orig_dist["ref"][field], bins=bins, density=True, alpha=0.55, color=ORIG_COLOR
        )
        n_gt_zs, _, _ = ax_gt.hist(
            zs_dist["ref"][field], bins=bins, density=True, alpha=0.55, color=ZS_COLOR
        )
        ax_gt.set_title(f"{field} — ground truth")

        ymax = max(n_pred_wt.max(), n_pred_zs.max(), n_gt_wt.max(), n_gt_zs.max()) * 1.05
        for ax in (ax_pred, ax_gt):
            ax.set_xlim(bins[0], bins[-1])
            ax.set_ylim(0, ymax)
            # Few enough ticks that labels never collide at this panel width.
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5, prune="both"))
            ax.spines[["top", "right"]].set_visible(False)

        p0, _ = np.histogram(orig_dist["pred"][field], bins=bins, density=True)
        p1, _ = np.histogram(zs_dist["pred"][field], bins=bins, density=True)
        l1_scores[field] = float(np.abs(p0 - p1).sum() * (bins[1] - bins[0]))

    axes[-1, 0].set_xlabel("value")
    axes[-1, 1].set_xlabel("value")
    h_label = f"first {hkey} frames" if hkey != "full" and hkey != "1-step" else hkey
    fig.suptitle(
        f"{RUN}\nckpt epoch {epoch} · split={SPLIT} · horizon={h_label}",
        y=1.02,
    )
    # A per-axes legend wider than its panel makes tight_layout shrink the axes box
    # to a sliver, so the datasets get one shared figure legend in reserved space.
    fig.tight_layout(rect=(0.0, 0.05, 1.0, 1.0))
    fig.subplots_adjust(hspace=0.45)
    fig.legend(
        handles=[
            Patch(facecolor=ORIG_COLOR, alpha=0.55, label=_dataset_label(ORIGINAL_DATA)),
            Patch(facecolor=ZS_COLOR, alpha=0.55, label=_dataset_label(DATA)),
        ],
        loc="lower center",
        ncol=2,
        frameon=False,
        fontsize=annotation_font_size(),
    )
    stem = "full" if hkey in ("full", "1-step") else f"t{hkey}"
    short_names = globals().get("SHORT_NAMES", {})
    run_stem = short_names.get(RUN, RUN)
    out = Path("figures") / f"zero_shot_hist_{run_stem}_{stem}_{DATA}.pdf"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches="tight")
    print(out.resolve())
    display(fig)
    plt.close(fig)

    print(f"horizon={h_label} — prediction histogram L1 (normalized dens.):")
    for field, l1 in l1_scores.items():
        print(f"  {field}: L1={l1:.4f}  (0 = identical dens.)")
    print()


## r.m.s. velocity: zero-shot predicted vs true

Same Fig. 3i overlay as in `analyze_checkpoints.ipynb`, but on **`DATA`** (not the
run's training set). Time 0 is GBE onset from the true RMS curve. It reads the rollout
cache; a GPU is needed only if this run/checkpoint/dataset is not cached yet.

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

apply_paper_style()

overrides = {"batch_size": BATCH_SIZE}
if N_STEPS_INPUT is not None:
    overrides["n_steps_input"] = N_STEPS_INPUT

flow_results = evaluate_flow_metrics(
    get_run(PROJECT, RUN, ENTITY),
    metric="rollout_valid",
    split="rollout_test",
    data=DATA,
    checkpoints_dir=CHECKPOINTS_DIR,
    data_overrides=overrides,
    max_rollout_steps=200,
    cache_dir=CACHE_DIR,
    refresh_cache=REFRESH_CACHE,
)
print(f"embryos ({DATA}): {len(flow_results)}")
for r in flow_results:
    print(
        f"  {r.file or r.batch}: GBE onset gt={r.gbe_onset_gt:.2f} / "
        f"pred={r.gbe_onset_pred:.2f} min (t0_omega_gt={r.t0_omega_gt:.2f}), "
        f"n={len(r.t)}, rms_gt peak={r.rms_gt.max():.2f}, "
        f"rms_pred peak={r.rms_pred.max():.2f}"
    )

fig, ax = plt.subplots(figsize=PANEL_SIZE)
plot_rms_velocity_overlay(
    flow_results,
    ax=ax,
    title=f"r.m.s. tissue velocity ({DATA})",
    ylim=(0.0, 7.0),
)
plt.show()


## Compare zero-shot across runs

For each name in `RUNS`, select the checkpoint with the lowest rollout-validation
**VRMSE**, evaluate **only** open-loop `rollout_test` on `DATA`, and collect a table.
The physical-unit trajectories are cached during that same pass, so all later RMS,
distribution, and video cells reuse them. Matching cached runs are read immediately.
Does not evaluate the original training test set. Runs with no checkpoints are skipped.

In [ ]:
zs_table = compare_zero_shot(
    PROJECT,
    RUNS,
    data=DATA,
    entity=ENTITY,
    full=FULL,
    make_videos=False,
    batch_size=BATCH_SIZE,
    n_steps_input=N_STEPS_INPUT,
    cache_dir=CACHE_DIR,
    refresh_cache=REFRESH_CACHE,
)
zs_table.round(4)

### VRMSE bar chart

One bar per model, coloured to match the in-distribution figure in
`analyze_checkpoints.ipynb`. Two PDFs: **full** zero-shot rollout-test VRMSE, and
the same metric on only the **first 30 predicted frames**. Both are scored from
the cached physical-unit trajectories (`compare_zero_shot` already wrote them),
so this cell does not run the model. Type 42 fonts keep the text editable in
Illustrator / Inkscape.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

# Type 42 fonts plus one font scale for every figure in the notebook. Raise
# FONT_SIZE to enlarge every text element (ticks, labels, titles, legends).
FONT_SIZE = 13.0
apply_paper_style(FONT_SIZE)

SHORT_NAMES = {
    "FFNO_morph_WT-morph-delta-FFNOW[--]-AdamW-0.0001": "FFNO",
    "PoseidonL_ft_morph_WT-morph-full-ScOTW[--]-AdamW-0.0001": "Poseidon-L",
    "SineNet_morph_WT-morph-delta-SineN[--]-AdamW-0.0001": "SineNet",
    "Walrus_crps_morph_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus CRPS",
    "Walrus_ft_morph_WT_no_myosin-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus",
    "Advection_morph_WT-morph-full-Advec[--]-AdamW-0.0001": "Advection",
    "MeanField_morph_WT-morph-full-MeanF[--]-AdamW-0.0001": "Mean-field",
    "FFNO_morph_WT_myosin-morph-delta-FFNOW[--]-AdamW-0.0001": "FFNO",
    "PoseidonL_ft_morph_WT_myosin-morph-full-ScOTW[--]-AdamW-0.0001": "Poseidon-L",
    "SineNet_morph_WT_myosin-morph-delta-SineN[--]-AdamW-0.0001": "SineNet",
    "Walrus_crps_morph_myosin_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus CRPS",
    "Walrus_ft_morph_WT_myosin-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus",
    "Advection_morph_WT_myosin-morph-full-Advec[--]-AdamW-0.0001": "Advection",
    "MeanField_morph_WT_myosin-morph-full-MeanF[--]-AdamW-0.0001": "Mean-field",
    "Walrus_scratch_morph_WT_myosin-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus (scratch)",
    "Walrus_scratch_morph_WT_no_myosin-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus (scratch)",
}
# Keyed by short name so colours stay put no matter how zs_table is ordered.
MODEL_COLORS = {
    "FFNO": "#4C78A8",
    "Poseidon-L": "#F58518",
    "SineNet": "#54A24B",
    "Walrus CRPS": "#B279A2",
    "Walrus": "#E45756",
    "Walrus (scratch)": "#9C755F",
    "Advection": "#72B7B2",
    "Mean-field": "#FF9DA6",
}
PLOT_ORDER = [
    "Walrus",
    "Walrus (scratch)",
    "Walrus CRPS",
    "Poseidon-L",
    "SineNet",
    "FFNO",
    "Mean-field",
    "Advection",
]
# Short names dropped from the VRMSE / spectral bars. Empty = show everyone.
PLOT_EXCLUDE = ["Mean-field", "Advection"]

# An early horizon plus the full cached trajectory. Both read the rollout cache
# written by compare_zero_shot — no GPU / no model reload.
VRMSE_HORIZONS = {
    "full": None,
    "t15": 15,
}

overrides = {"batch_size": BATCH_SIZE}
if N_STEPS_INPUT is not None:
    overrides["n_steps_input"] = N_STEPS_INPUT

horizon_scores = {key: {} for key in VRMSE_HORIZONS}
for run_name in zs_table.index:
    rollouts = cached_rollouts(
        get_run(PROJECT, run_name, ENTITY),
        data=DATA,
        data_overrides=overrides,
        split="rollout_test",
        checkpoints_dir=CHECKPOINTS_DIR,
        cache_dir=CACHE_DIR,
        refresh_cache=REFRESH_CACHE,
    )
    short = SHORT_NAMES.get(run_name, run_name)
    for key, n_frames in VRMSE_HORIZONS.items():
        horizon_scores[key][short] = vrmse_from_rollouts(rollouts, n_frames=n_frames)

epochs = {
    SHORT_NAMES.get(run, run): int(v)
    for run, v in zs_table["ckpt_epoch"].items()
}


def _ordered_plot_names(scores):
    names = [n for n in PLOT_ORDER if n in scores] + [
        n for n in scores if n not in PLOT_ORDER
    ]
    skip = set(PLOT_EXCLUDE)
    return [n for n in names if n not in skip]


def _plot_vrmse_bars(scores, *, horizon_label, stem):
    names = _ordered_plot_names(scores)
    y = np.array([scores[n] for n in names])
    x = np.arange(len(names))

    # One panel of the same size as a spectral subplot (see the cell below).
    fig, axes = panel_figure(1)
    ax = axes[0]
    bars = ax.bar(
        x,
        y,
        0.62,
        color=[MODEL_COLORS.get(n, "0.6") for n in names],
        edgecolor="0.15",
        linewidth=0.6,
        zorder=3,
    )
    for rect in bars:
        h = rect.get_height()
        ax.annotate(
            f"{h:.2f}",
            xy=(rect.get_x() + rect.get_width() / 2, h),
            xytext=(0, 2),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=annotation_font_size(),
            color="0.25",
        )

    ax.set_xticks(x, names, rotation=30, ha="right")
    ax.set_ylabel("Rollout-test VRMSE")
    ax.set_ylim(0, 3)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title(horizon_label)
    # Above the reserved top margin, so a long dataset name never shrinks the axes.
    fig.suptitle(f"Zero-shot VRMSE (lower is better)\n{DATA}", y=1.0, va="bottom")

    out = Path("figures") / f"zero_shot_vrmse_{stem}_{DATA}.pdf"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches="tight")
    print(out.resolve())
    plt.show()


_plot_vrmse_bars(horizon_scores["full"], horizon_label="Full trajectory", stem="full")
_plot_vrmse_bars(
    horizon_scores["t15"],
    horizon_label="First 15 predicted frames",
    stem="t15",
)
# Kept out of the figures so both panels stay the same size; belongs in the caption.
print("checkpoints selected by rollout-validation VRMSE on the WT val set")


### Spectral-error comparison

The same model comparison using The Well's binned spectral metric. Each panel shows
normalized spectral MSE in the default low-, medium-, or high-wavenumber bin. We
report normalized rather than raw spectral MSE because raw energy is not comparable
across velocity and myosin fields. Scores are recomputed from the cached physical-unit
rollouts, for both the full trajectory and the first 30 predicted frames; no model
inference is required on a cache hit.

In [ ]:
# The Well's default spectral bins are approximately log-spaced from zero to
# the spatial Nyquist wavenumber. Scores are averaged over frames, fields,
# embryos, and then datasets, matching the aggregation used for VRMSE above.
SPECTRAL_HORIZONS = {
    "full": None,
    "t15": 15,
}
SPECTRAL_BIN_LABELS = ["Low frequency", "Medium frequency", "High frequency"]
# Shared y-limit across the three bins. Bars above it are clipped, so their value
# label is drawn inside the bar instead of above the axes where it would be cropped.
SPECTRAL_YMAX = 3.0

overrides = {"batch_size": BATCH_SIZE}
if N_STEPS_INPUT is not None:
    overrides["n_steps_input"] = N_STEPS_INPUT

spectral_scores = {key: {} for key in SPECTRAL_HORIZONS}
for run_name in zs_table.index:
    rollouts = cached_rollouts(
        get_run(PROJECT, run_name, ENTITY),
        data=DATA,
        data_overrides=overrides,
        split="rollout_test",
        checkpoints_dir=CHECKPOINTS_DIR,
        cache_dir=CACHE_DIR,
        refresh_cache=REFRESH_CACHE,
    )
    short = SHORT_NAMES.get(run_name, run_name)
    for horizon_key, n_frames in SPECTRAL_HORIZONS.items():
        metrics = spectral_metrics_from_rollouts(rollouts, n_frames=n_frames)
        spectral_scores[horizon_key][short] = [
            metrics[f"spectral_error_nmse_per_bin_{i}"] for i in range(3)
        ]


def _plot_spectral_bars(scores, *, horizon_label, stem):
    names = _ordered_plot_names(scores)
    values = np.asarray([scores[name] for name in names])

    fig, axes = panel_figure(3, sharex=True)
    x = np.arange(len(names))
    for bin_idx, (ax, bin_label) in enumerate(zip(axes, SPECTRAL_BIN_LABELS)):
        y = values[:, bin_idx]
        bars = ax.bar(
            x,
            y,
            0.62,
            color=[MODEL_COLORS.get(name, "0.6") for name in names],
            edgecolor="0.15",
            linewidth=0.6,
            zorder=3,
        )
        ax.set_ylim(0, SPECTRAL_YMAX)
        for rect, value in zip(bars, y):
            clipped = value > SPECTRAL_YMAX
            ax.annotate(
                f"{value:.2f}",
                xy=(rect.get_x() + rect.get_width() / 2, min(value, SPECTRAL_YMAX)),
                xytext=(0, -4 if clipped else 3),
                textcoords="offset points",
                ha="center",
                va="top" if clipped else "bottom",
                fontsize=annotation_font_size(),
                color="white" if clipped else "0.25",
            )
        ax.set_title(bin_label)
        ax.set_xticks(x, names, rotation=30, ha="right")
        ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    axes[0].set_ylabel("Normalized spectral MSE")
    fig.suptitle(
        f"Zero-shot spectral error, {horizon_label.lower()} (lower is better)\n{DATA}",
        y=1.0,
        va="bottom",
    )

    out = Path("figures") / f"zero_shot_spectral_nmse_{stem}_{DATA}.pdf"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches="tight")
    print(out.resolve())
    plt.show()


_plot_spectral_bars(
    spectral_scores["full"], horizon_label="Full trajectory", stem="full"
)
_plot_spectral_bars(
    spectral_scores["t15"], horizon_label="First 15 predicted frames", stem="t15"
)

### Paper-style normalized flow residual (SI Eq. 6)

This section approximates the flow-forecast evaluation in arXiv:2405.18382. For each
frame it computes the spatially averaged normalized residual

$$R(t)=1-\frac{\langle\hat{\mathbf v}(t)\cdot\mathbf v(t)\rangle}
{\sqrt{\langle|\hat{\mathbf v}(t)|^2\rangle\langle|\mathbf v(t)|^2\rangle}}.$$

It measures flow **pattern and direction**, not magnitude: positively rescaling either
field leaves $R$ unchanged. We report it alongside, not instead of, magnitude-sensitive
VRMSE.

For a controlled temporal comparison, every model uses its native context ending at
ventral-furrow onset (`t=0`) and is scored only on genuine open-loop predictions from
`+1` through `+15` or `+20` min. All models use the same intersection of eligible WT
test embryos. The paper's neural network instead receives a single myosin snapshot at
`t=0`; its reported 6% average-test and 7% example-trajectory values are therefore
contextual references, not directly equivalent targets. The paper does not specify
whether `t=0` enters its average, its exact held-out size or error-bar definition, or
whether the PCA crop/mask was reused for NN scoring, so this analysis does not invent
those choices.

The first execution creates a separate `rollout_test_vf_aligned` cache and requires
inference. Later reruns and plot edits are CPU-only cache hits.

In [ ]:
from pathlib import Path

# This is a new, deliberately VF-aligned inference protocol. The first run creates
# distinct caches; subsequent runs reuse them. Fractions are converted to percent only
# for display, matching the convention in the paper.
PAPER_HORIZONS = (15, 20)
paper_data_overrides = {}
if N_STEPS_INPUT is not None:
    paper_data_overrides["n_steps_input"] = N_STEPS_INPUT
paper_result = compare_paper_residual(
    PROJECT,
    RUNS,
    data=DATA,
    entity=ENTITY,
    horizon_minutes=max(PAPER_HORIZONS),
    horizons=PAPER_HORIZONS,
    batch_size=BATCH_SIZE,
    keep_n_steps_input=N_STEPS_INPUT is None,
    checkpoints_dirs=(
        {run_name: CHECKPOINTS_DIR for run_name in RUNS}
        if CHECKPOINTS_DIR is not None
        else None
    ),
    cache_dir=CACHE_DIR,
    refresh_cache=REFRESH_CACHE,
    include_paper_mean_field=True,
    **paper_data_overrides,
)

cohort = paper_result["cohort"]
print(
    f"Common VF-aligned cohort: {len(cohort.files)} embryos; "
    f"largest native context: {cohort.max_context} frames"
)
for path in cohort.files:
    print("  included:", path.name)
for name, reason in cohort.excluded.items():
    print("  excluded:", name, "—", reason)

paper_table = paper_result["table"].copy()
percent_columns = [c for c in paper_table if c.startswith("residual_t") and not c.endswith("_n")]
paper_table[percent_columns] *= 100.0
display(paper_table.round(2))

PAPER_MEAN_KEY = "Paper mean-field (entire dataset)"
PAPER_MEAN_LABEL = "Mean-field (paper def.)"
PAPER_COLORS = {**MODEL_COLORS, PAPER_MEAN_LABEL: "#9D755D"}


def _paper_label(run_name):
    if run_name == PAPER_MEAN_KEY:
        return PAPER_MEAN_LABEL
    return SHORT_NAMES.get(run_name, run_name)


# Time-resolved residual after VF. Shading is ± one SD across the same embryos.
fig, axes = panel_figure(1)
ax = axes[0]
for run_name, aggregate in paper_result["aggregates"].items():
    label = _paper_label(run_name)
    color = PAPER_COLORS.get(label, "0.45")
    mean = 100.0 * aggregate.mean
    std = 100.0 * aggregate.std
    ax.plot(aggregate.times, mean, label=label, color=color, linewidth=2)
    ax.fill_between(aggregate.times, mean - std, mean + std, color=color, alpha=0.10)
ax.axhline(6.0, color="0.25", linestyle="--", linewidth=1.0, label="Paper: 6% avg. test")
ax.axhline(7.0, color="0.5", linestyle=":", linewidth=1.0, label="Paper: 7% example trajectory")
ax.set(xlabel="Minutes after VF onset", ylabel="Normalized flow residual (%)", xlim=(1, 20))
ax.set_ylim(bottom=0)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=annotation_font_size(), ncol=2)
ax.set_title(f"VF-aligned paper-style flow residual\n{DATA} (N={len(cohort.files)})")
curve_out = Path("figures") / f"paper_residual_curve_{DATA}.pdf"
curve_out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(curve_out, bbox_inches="tight")
print(curve_out.resolve())
plt.show()


def _plot_paper_residual_bars(horizon):
    scores = {
        _paper_label(name): aggregate.horizon_scores[horizon]
        for name, aggregate in paper_result["aggregates"].items()
    }
    stds = {
        _paper_label(name): aggregate.horizon_std[horizon]
        for name, aggregate in paper_result["aggregates"].items()
    }
    order = PLOT_ORDER + [PAPER_MEAN_LABEL]
    names = [name for name in order if name in scores] + [
        name for name in scores if name not in order
    ]
    values = 100.0 * np.array([scores[name] for name in names])
    errors = 100.0 * np.array([stds[name] for name in names])
    x = np.arange(len(names))

    fig, axes = panel_figure(1)
    ax = axes[0]
    bars = ax.bar(
        x,
        values,
        0.62,
        yerr=errors,
        capsize=3,
        color=[PAPER_COLORS.get(name, "0.6") for name in names],
        edgecolor="0.15",
        linewidth=0.6,
        zorder=3,
    )
    for rect, value in zip(bars, values):
        ax.annotate(
            f"{value:.1f}%",
            (rect.get_x() + rect.get_width() / 2, value),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=annotation_font_size(),
        )
    ax.axhline(6.0, color="0.25", linestyle="--", linewidth=1.0, label="Paper: 6% avg. test")
    ax.axhline(7.0, color="0.5", linestyle=":", linewidth=1.0, label="Paper: 7% example trajectory")
    ax.set_xticks(x, names, rotation=30, ha="right")
    ax.set_ylabel("Mean normalized flow residual (%)")
    upper = min(200.0, max(30.0, np.ceil(np.nanmax(values + errors) * 1.15 / 10.0) * 10.0))
    ax.set_ylim(0, upper)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=annotation_font_size())
    ax.set_title(f"VF +1 to +{horizon} min (N={len(cohort.files)})")
    fig.suptitle(f"Paper-style flow residual (lower is better)\n{DATA}", y=1.0, va="bottom")
    out = Path("figures") / f"paper_residual_t{horizon}_{DATA}.pdf"
    fig.savefig(out, bbox_inches="tight")
    print(out.resolve())
    plt.show()


for horizon in PAPER_HORIZONS:
    _plot_paper_residual_bars(horizon)

print(
    "Protocol note: paper input = one myosin snapshot at VF; our models use their "
    "native velocity or velocity+myosin context ending at VF. Paper grid = 236×200; "
    "ours = 64×96. The paper does not document applying its PCA mask to NN scoring. "
    "The paper-defined mean field averages the full dataset and is therefore not a "
    "leakage-free learned baseline."
)

### r.m.s. velocity: all runs overlaid

Same Fig. 3i curve as above, but with every run in `RUNS` on one axes and a single
black ground-truth curve. Each coloured line is that run's ensemble mean over the
test embryos; the grey band is the ground-truth spread (± s.d. across embryos).

All runs see the same embryos, so their ground-truth curves should coincide — they
are pooled into one reference and you get a warning if any run disagrees, which
would mean a mismatched `n_steps_input` or dataset. This reads the trajectories saved
by `compare_zero_shot`; changing plot style or labels does not rerun inference.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

%matplotlib inline

apply_paper_style()

SHOW_INDIVIDUAL_EMBRYOS = False  # thin per-embryo traces behind the means
SHOW_RUN_BANDS = False  # ± s.d. band per run (busy with many runs)

# One rollout per run; SHORT_NAMES / MODEL_COLORS come from the bar-chart cell above.
flow_by_run = compare_flow_metrics(
    PROJECT,
    RUNS,
    data=DATA,
    entity=ENTITY,
    batch_size=BATCH_SIZE,
    n_steps_input=N_STEPS_INPUT,
    max_rollout_steps=200,
    cache_dir=CACHE_DIR,
    refresh_cache=REFRESH_CACHE,
)
SHORT_NAMES = {
    "FFNO_morph_WT-morph-delta-FFNOW[--]-AdamW-0.0001": "FFNO",
    "PoseidonL_ft_morph_WT-morph-full-ScOTW[--]-AdamW-0.0001": "Poseidon-L",
    "SineNet_morph_WT-morph-delta-SineN[--]-AdamW-0.0001": "SineNet",
    "Walrus_crps_morph_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus CRPS",
    "Walrus_ft_morph_WT_no_myosin-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus (det.)",
}

MODEL_COLORS = {
    "FFNO": "#4C78A8",
    "Poseidon-L": "#F58518",
    "SineNet": "#54A24B",
    "Walrus CRPS": "#B279A2",
    "Walrus (det.)": "#E45756",
    "Advection": "#72B7B2",
    "Mean-field": "#FF9DA6",
}

for run, results in flow_by_run.items():
    print(f"{SHORT_NAMES.get(run, run)}: {len(results)} embryos")
    for r in results:
        print(
            f"  {r.file or r.batch}: GBE onset gt={r.gbe_onset_gt:.2f} / "
            f"pred={r.gbe_onset_pred:.2f} min, n={len(r.t)}, "
            f"rms_gt peak={r.rms_gt.max():.2f}, rms_pred peak={r.rms_pred.max():.2f}"
        )

fig, ax = plt.subplots(figsize=PANEL_SIZE)
plot_rms_velocity_multi_run(
    flow_by_run,
    ax=ax,
    labels={run: SHORT_NAMES.get(run, run) for run in flow_by_run},
    colors=MODEL_COLORS,
    show_individuals=SHOW_INDIVIDUAL_EMBRYOS,
    show_run_bands=SHOW_RUN_BANDS,
    title=f"r.m.s. tissue velocity — zero-shot on {DATA}",
    ylim=(0.0, 7.0),
)
fig.tight_layout()

out = Path("figures") / f"rms_velocity_multi_run_{DATA}.pdf"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, bbox_inches="tight")
print(out.resolve())
plt.show()

## Tips

- Swap `RUN` / `DATA` and re-run the peek + eval cells — no new shell scripts needed.
- Add a new dataset by dropping a yaml in `walrus/configs/data/` (see `morphogenesis_WT.yaml`).
- For a quick smoke test set `FULL = False` and/or `MAKE_VIDEOS = False`.
- If the run used globalnorm, the target Well root needs a `stats.yaml`.

## r.m.s. velocity across rearing temperatures

Two Fig. 3i-style panels, one curve per temperature:

1. **ground truth only** — measured r.m.s. velocity at each temperature
2. **predictions only** — one model's open-loop rollout at each temperature

The ground-truth curves come from the data, not the model, so a single pass per
dataset produces both panels. Time is relative to the ground-truth GBE onset in both,
so the two panels share an x axis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

%matplotlib inline

apply_paper_style()

# Model whose predictions go in the second panel (ground truth is model-independent).
TEMP_RUN = "Walrus_crps_morph_myosin_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001"
TEMP_RUN_LABEL = "Walrus CRPS"

# Add "22° C": "morphogenesis_WT" if you want the baseline WT set as the middle curve.
# The WT files carry no temperature tag, so that mapping is an assumption, not metadata.
TEMP_DATASETS = {
    "17° C": "morphogenesis_WT_17_degrees_myosin",
    "27° C": "morphogenesis_WT_27_degrees_myosin",
}
TEMP_COLORS = {"17° C": "#3C6DA8", "22° C": "#2B2B2B", "27° C": "#A6413C"}

TEMP_YLIM = (0.0, 7.0)
SHOW_TEMP_BANDS = True  # ± s.d. across embryos at that temperature
FIGDIR = Path("figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

overrides = {"batch_size": BATCH_SIZE}
if N_STEPS_INPUT is not None:
    overrides["n_steps_input"] = N_STEPS_INPUT

# One rollout per temperature; each result carries both the GT and predicted curve.
temp_run = get_run(PROJECT, TEMP_RUN, ENTITY)
flow_by_temp = {}
for label, data_name in TEMP_DATASETS.items():
    flow_by_temp[label] = evaluate_flow_metrics(
        temp_run,
        metric="rollout_valid",
        split="rollout_test",
        data=data_name,
        checkpoints_dir=CHECKPOINTS_DIR,
        data_overrides=overrides,
        max_rollout_steps=200,
        cache_dir=CACHE_DIR,
        refresh_cache=REFRESH_CACHE,
    )
    print(f"{label} ({data_name}): {len(flow_by_temp[label])} embryos")
    for r in flow_by_temp[label]:
        print(
            f"  {r.file or r.batch}: n={len(r.t)}, "
            f"rms_gt peak={r.rms_gt.max():.2f}, rms_pred peak={r.rms_pred.max():.2f}"
        )

# --- panel 1: ground truth only ---
fig, ax = plt.subplots(figsize=PANEL_SIZE)
plot_rms_velocity_by_group(
    flow_by_temp,
    which="gt",
    ax=ax,
    colors=TEMP_COLORS,
    show_bands=SHOW_TEMP_BANDS,
    title="r.m.s. tissue velocity — ground truth",
    ylim=TEMP_YLIM,
)
fig.tight_layout()
gt_out = FIGDIR / "rms_velocity_ground_truth_by_temperature.pdf"
fig.savefig(gt_out, bbox_inches="tight")
print(gt_out.resolve())
plt.show()

# --- panel 2: predictions only ---
fig, ax = plt.subplots(figsize=PANEL_SIZE)
plot_rms_velocity_by_group(
    flow_by_temp,
    which="pred",
    ax=ax,
    colors=TEMP_COLORS,
    show_bands=SHOW_TEMP_BANDS,
    title=f"r.m.s. tissue velocity — {TEMP_RUN_LABEL} predicted",
    ylim=TEMP_YLIM,
)
fig.tight_layout()
pred_out = FIGDIR / "rms_velocity_predicted_by_temperature.pdf"
fig.savefig(pred_out, bbox_inches="tight")
print(pred_out.resolve())
plt.show()

## r.m.s. velocity across mutations

Same two-panel layout as the temperature plots, one curve per genotype:

1. **ground truth** (model-independent)
2. **open-loop predictions** for `MUTATION_RUN`

Time 0 is GBE onset from the true RMS curve. Cached `rollout_test` trajectories are reused when present.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

%matplotlib inline

apply_paper_style()

# Model whose predictions go in the second panel (ground truth is model-independent).
MUTATION_RUN = "Walrus_crps_morph_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001"
MUTATION_RUN_LABEL = "Walrus CRPS"

MUTATION_DATASETS = {
    "even skipped": "morphogenesis_even_skipped_r13",
    "spaetzle A": "morphogenesis_spaetzle_a",
    "halo twist": "morphogenesis_halo_twist_ey53",
}
MUTATION_COLORS = {
    "even skipped": "#54A24B",
    "spaetzle A": "#B279A2",
    "halo twist": "#F58518",
}

MUTATION_YLIM = (0.0, 7.0)
# Halo twist movies mostly start ~30 min before GBE onset, so the window every halo
# embryo shares is only ~10 min. "union" averages over whichever embryos cover each
# time instead, keeping the full pre-onset ramp.
MUTATION_TIME_COVERAGE = "union"
MUTATION_XLIM = (-40.0, 80.0)
SHOW_MUTATION_BANDS = True  # ± s.d. across embryos of that genotype
FIGDIR = Path("figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

overrides = {"batch_size": BATCH_SIZE}
if N_STEPS_INPUT is not None:
    overrides["n_steps_input"] = N_STEPS_INPUT

mutation_run = get_run(PROJECT, MUTATION_RUN, ENTITY)
flow_by_mutation = {}
for label, data_name in MUTATION_DATASETS.items():
    flow_by_mutation[label] = evaluate_flow_metrics(
        mutation_run,
        metric="rollout_valid",
        split="rollout_test",
        data=data_name,
        checkpoints_dir=CHECKPOINTS_DIR,
        data_overrides=overrides,
        max_rollout_steps=200,
        cache_dir=CACHE_DIR,
        refresh_cache=REFRESH_CACHE,
    )
    print(f"{label} ({data_name}): {len(flow_by_mutation[label])} embryos")
    for r in flow_by_mutation[label]:
        print(
            f"  {r.file or r.batch}: n={len(r.t)}, "
            f"rms_gt peak={r.rms_gt.max():.2f}, rms_pred peak={r.rms_pred.max():.2f}"
        )

fig, ax = plt.subplots(figsize=PANEL_SIZE)
plot_rms_velocity_by_group(
    flow_by_mutation,
    which="gt",
    ax=ax,
    colors=MUTATION_COLORS,
    show_bands=SHOW_MUTATION_BANDS,
    title="r.m.s. tissue velocity — ground truth",
    ylim=MUTATION_YLIM,
    xlim=MUTATION_XLIM,
    time_coverage=MUTATION_TIME_COVERAGE,
)
fig.tight_layout()
gt_out = FIGDIR / "rms_velocity_ground_truth_by_mutation.pdf"
fig.savefig(gt_out, bbox_inches="tight")
print(gt_out.resolve())
plt.show()

fig, ax = plt.subplots(figsize=PANEL_SIZE)
plot_rms_velocity_by_group(
    flow_by_mutation,
    which="pred",
    ax=ax,
    colors=MUTATION_COLORS,
    show_bands=SHOW_MUTATION_BANDS,
    title=f"r.m.s. tissue velocity — {MUTATION_RUN_LABEL} predicted",
    ylim=MUTATION_YLIM,
    xlim=MUTATION_XLIM,
    time_coverage=MUTATION_TIME_COVERAGE,
)
fig.tight_layout()
pred_out = FIGDIR / "rms_velocity_predicted_by_mutation.pdf"
fig.savefig(pred_out, bbox_inches="tight")
print(pred_out.resolve())
plt.show()


## Tips

- Swap `RUN` / `DATA` and re-run the peek + eval cells — no new shell scripts needed.
- Add a new dataset by dropping a yaml in `walrus/configs/data/` (see `morphogenesis_WT.yaml`).
- For a quick smoke test set `FULL = False` and/or `MAKE_VIDEOS = False`.
- If the run used globalnorm, the target Well root needs a `stats.yaml`.